In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
data_path=Path("../../data")
out_path=data_path/"clean_data_ivana"
edu=pd.read_csv(data_path/"edu_years.csv")

In [13]:
edu.head()
edu.columns
edu.info()
print(edu.shape)

<class 'pandas.DataFrame'>
RangeIndex: 46035 entries, 0 to 46034
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   STRUCTURE               46035 non-null  str    
 1   STRUCTURE_ID            46035 non-null  str    
 2   ACTION                  46035 non-null  str    
 3   FREQ                    46035 non-null  str    
 4   FREQ_LABEL              46035 non-null  str    
 5   REF_AREA                46035 non-null  str    
 6   REF_AREA_LABEL          46035 non-null  str    
 7   INDICATOR               46035 non-null  str    
 8   INDICATOR_LABEL         46035 non-null  str    
 9   SEX                     46035 non-null  str    
 10  SEX_LABEL               46035 non-null  str    
 11  AGE                     46035 non-null  str    
 12  AGE_LABEL               46035 non-null  str    
 13  URBANISATION            46035 non-null  str    
 14  URBANISATION_LABEL      46035 non-null  str    
 

In [14]:
edu[["REF_AREA", "REF_AREA_LABEL", "INDICATOR", "SEX", "AGE",
     "URBANISATION", "TIME_PERIOD", "OBS_VALUE"]].head(10)

,REF_AREA,REF_AREA_LABEL,INDICATOR,SEX,AGE,URBANISATION,TIME_PERIOD,OBS_VALUE
0,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1970,2.0
1,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1970,3.0
2,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1970,5.0
3,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1970,6.0
4,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1971,2.0
5,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1971,3.0
6,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1971,5.0
7,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1971,6.0
8,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1972,2.0
9,ABW,Aruba,WB_EDSTATS_SE_DURS,_T,_T,_T,1972,3.0


In [18]:
print(edu["INDICATOR"].unique())
print(edu["SEX"].unique())
print(edu["AGE"].unique())
print(edu["URBANISATION"].unique())

<ArrowStringArray>
['WB_EDSTATS_SE_DURS']
Length: 1, dtype: str
<ArrowStringArray>
['_T']
Length: 1, dtype: str
<ArrowStringArray>
['_T']
Length: 1, dtype: str
<ArrowStringArray>
['_T']
Length: 1, dtype: str


In [19]:
splits = edu.groupby(["REF_AREA", "TIME_PERIOD"]).nunique().max()
splits[splits > 1]

UNIT_MEASURE              2
UNIT_MEASURE_LABEL        2
COMP_BREAKDOWN_1          4
COMP_BREAKDOWN_1_LABEL    4
OBS_VALUE                 4
OBS_STATUS                2
OBS_STATUS_LABEL          2
dtype: int64

In [20]:
edu[["COMP_BREAKDOWN_1", "COMP_BREAKDOWN_1_LABEL", "UNIT_MEASURE_LABEL"]].drop_duplicates()

,COMP_BREAKDOWN_1,COMP_BREAKDOWN_1_LABEL,UNIT_MEASURE_LABEL
0,ISCED11_2,Education: Lower Secondary,Years
1,ISCED11_3,Education: Upper Secondary,Years
2,ISCED11_2_3,Education: Secondary,Years
3,_Z,Not Applicable,Number


In [ ]:
# combining two types of secondary education into total

edu_clean = edu[(edu["COMP_BREAKDOWN_1"] == "ISCED11_2_3") &
                (edu["OBS_STATUS"] == "A")]

In [23]:
edu_clean = (edu_clean[["REF_AREA", "REF_AREA_LABEL", "TIME_PERIOD", "OBS_VALUE"]]
             .rename(columns={"REF_AREA": "iso3",
                              "REF_AREA_LABEL": "country",
                              "TIME_PERIOD": "year",
                              "OBS_VALUE": "secondary_edu_years"}))

In [24]:
edu_clean.head(10)

,iso3,country,year,secondary_edu_years
2,ABW,Aruba,1970,5.0
6,ABW,Aruba,1971,5.0
10,ABW,Aruba,1972,5.0
14,ABW,Aruba,1973,5.0
18,ABW,Aruba,1974,5.0
22,ABW,Aruba,1975,5.0
26,ABW,Aruba,1976,5.0
30,ABW,Aruba,1977,5.0
34,ABW,Aruba,1978,5.0
38,ABW,Aruba,1979,5.0


In [25]:
edu_clean = edu_clean.reset_index(drop=True)

In [27]:
# checks
print(edu_clean.shape)
print(edu_clean.duplicated(["iso3", "year"]).sum())   
print(edu_clean.isna().sum())

(10994, 4)
0
iso3                   0
country                0
year                   0
secondary_edu_years    0
dtype: int64


In [28]:
edu_clean.to_csv(out_path / "edu_clean.csv", index=False)